In [0]:
CATALOG = "energy_puls"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

In [0]:

display(spark.sql(f"""
  SELECT numero_dpe, COUNT(*) AS nb_versions
  FROM {BRONZE}.dpe_raw
  GROUP BY 1 HAVING COUNT(*) > 1
  ORDER BY nb_versions DESC LIMIT 10
"""))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy("numero_dpe").orderBy(F.col("date_etablissement").desc())

clean = (spark.table(f"{BRONZE}.dpe_raw")
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn")
  
    .withColumn("classe_dpe", F.upper(F.trim("classe_dpe"))))

bad = clean.filter("surface_habitable <= 0 OR code_commune NOT RLIKE '^[0-9]{5}$'")
good = clean.filter("surface_habitable > 0 AND code_commune RLIKE '^[0-9]{5}$'")

good.write.mode("overwrite").saveAsTable(f"{SILVER}.dpe_clean")
bad.write.mode("overwrite").saveAsTable(f"{SILVER}.dpe_quarantine")

print("Bronze     :", spark.table(f"{BRONZE}.dpe_raw").count())
print("Silver good:", good.count())
print("Quarantine :", bad.count())